# D383 — E-commerce Star Schema in Snowflake

This notebook builds a complete, runnable dimensional model for e-commerce analytics. It uses only Snowflake SQL and inline sample data—no files, stages, streams, or staging schema.

## WE DON'T HAVE CLI, ACCESS TO NOTEBOOKS IN SNOWFLAKE, PLS COPY PASTE AND RUN IT YOURSELF. MINDFUL OF SCHEMA, MAP TO YOURS PLS

## Learning goals

- Understand what a star schema is and why analytics systems use it.
- Declare the grain before creating the fact table.
- Separate descriptive dimensions from numeric business events.
- Use surrogate keys, conformed dimensions, role-playing dates, and degenerate dimensions.
- Build and query a complete Snowflake star schema.
- Validate key uniqueness, referential integrity, and additive measures.

> Run the cells in order in a Snowflake Notebook. SQL code cells use Snowflake Notebook's `%%sql` cell magic. The setup creates `ECOMMERCE_STAR_DB`, schema `ANALYTICS`, and a small warehouse. Change those names if your role cannot create them.

## 1. What is a star schema?

A **star schema** is a dimensional model with one central fact table joined directly to several denormalized dimension tables.

```text
                         DATE_DIM
                    (order/ship dates)
                              |
REGION_DIM --- CUSTOMER_DIM   |   PRODUCT_DIM
 (reused)          \          |          /
                    \         |         /
                     FACT_ORDER_ITEM
                    /         |         \
                   /          |          \
        WAREHOUSE_DIM     REGION_DIM     promotion/order numbers
                         (reused roles)   stored in the fact
```

The fact table answers **what happened and by how much**. Dimensions answer **who, what, when, and where**.

### The declared grain

The grain is: **one row for one product line on one customer order, fulfilled by one warehouse**.

`order_number + order_line_number` therefore identifies a business event. An order containing three products produces three fact rows. Every measure must be valid at this grain:

- `quantity`, `gross_amount`, `discount_amount`, `net_amount`, `tax_amount`, `shipping_amount`, and `total_amount` are line-level facts.
- `unit_price` and `unit_cost` are captured at transaction time so history does not change when current product prices change.
- `order_count` uses `COUNT(DISTINCT order_number)`, while units use `SUM(quantity)`.

### Design choices

- Surrogate integer keys isolate analytics from operational identifiers.
- `DATE_DIM` is a conformed dimension played in two roles: order date and ship date.
- `REGION_DIM` is reused in two roles: customer region and warehouse region. The fact table joins directly to both roles, preserving a star-shaped query path.
- Category is kept inside `PRODUCT_DIM`. Splitting category into another table would create a snowflake schema.
- Order number and line number are **degenerate dimensions**: useful transaction identifiers stored in the fact because they do not need their own descriptive table.
- Customer and product business keys remain in dimensions for traceability.

## 2. Create the Snowflake environment

In [ ]:
%%sql
USE ROLE SYSADMIN;

CREATE WAREHOUSE IF NOT EXISTS ECOMMERCE_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

CREATE DATABASE IF NOT EXISTS ECOMMERCE_STAR_DB;
CREATE SCHEMA IF NOT EXISTS ECOMMERCE_STAR_DB.ANALYTICS;

USE WAREHOUSE ECOMMERCE_WH;
USE DATABASE ECOMMERCE_STAR_DB;
USE SCHEMA ANALYTICS;

## 3. Create dimension tables

Dimensions contain labels and attributes used to filter, group, and describe facts. Numeric surrogate keys are stable join keys. The `-1` rows inserted later represent **Unknown / Not Yet Available**, so a fact can always retain a valid dimension key.

Snowflake records primary and foreign keys on standard tables as metadata; applications and load processes must still validate them. The quality checks near the end demonstrate that validation.

In [ ]:
%%sql
CREATE OR REPLACE TABLE DATE_DIM (
    DATE_KEY             NUMBER(8,0) NOT NULL,
    FULL_DATE            DATE NOT NULL,
    DAY_OF_WEEK_NUMBER   NUMBER(1,0) NOT NULL,
    DAY_NAME             VARCHAR(10) NOT NULL,
    DAY_OF_MONTH         NUMBER(2,0) NOT NULL,
    DAY_OF_YEAR          NUMBER(3,0) NOT NULL,
    WEEK_OF_YEAR         NUMBER(2,0) NOT NULL,
    MONTH_NUMBER         NUMBER(2,0) NOT NULL,
    MONTH_NAME           VARCHAR(10) NOT NULL,
    QUARTER_NUMBER       NUMBER(1,0) NOT NULL,
    QUARTER_NAME         VARCHAR(2) NOT NULL,
    YEAR_NUMBER          NUMBER(4,0) NOT NULL,
    IS_WEEKEND           BOOLEAN NOT NULL,
    CONSTRAINT PK_DATE_DIM PRIMARY KEY (DATE_KEY)
);

CREATE OR REPLACE TABLE REGION_DIM (
    REGION_KEY           NUMBER(10,0) NOT NULL,
    COUNTRY_CODE         VARCHAR(2) NOT NULL,
    COUNTRY_NAME         VARCHAR(100) NOT NULL,
    STATE_CODE           VARCHAR(10) NOT NULL,
    STATE_NAME           VARCHAR(100) NOT NULL,
    SALES_REGION         VARCHAR(50) NOT NULL,
    CONSTRAINT PK_REGION_DIM PRIMARY KEY (REGION_KEY)
);

CREATE OR REPLACE TABLE CUSTOMER_DIM (
    CUSTOMER_KEY         NUMBER(10,0) NOT NULL,
    CUSTOMER_ID          VARCHAR(30) NOT NULL,
    CUSTOMER_NAME        VARCHAR(150) NOT NULL,
    EMAIL                VARCHAR(200),
    CUSTOMER_SEGMENT     VARCHAR(30) NOT NULL,
    CUSTOMER_REGION_KEY  NUMBER(10,0) NOT NULL,
    EFFECTIVE_FROM       DATE NOT NULL,
    EFFECTIVE_TO         DATE NOT NULL,
    IS_CURRENT           BOOLEAN NOT NULL,
    CONSTRAINT PK_CUSTOMER_DIM PRIMARY KEY (CUSTOMER_KEY),
    CONSTRAINT FK_CUSTOMER_REGION FOREIGN KEY (CUSTOMER_REGION_KEY)
      REFERENCES REGION_DIM(REGION_KEY)
);

CREATE OR REPLACE TABLE PRODUCT_DIM (
    PRODUCT_KEY          NUMBER(10,0) NOT NULL,
    PRODUCT_ID           VARCHAR(30) NOT NULL,
    SKU                  VARCHAR(40) NOT NULL,
    PRODUCT_NAME         VARCHAR(200) NOT NULL,
    BRAND_NAME           VARCHAR(100) NOT NULL,
    CATEGORY_ID          VARCHAR(30) NOT NULL,
    CATEGORY_NAME        VARCHAR(100) NOT NULL,
    SUBCATEGORY_NAME     VARCHAR(100) NOT NULL,
    CURRENT_LIST_PRICE   NUMBER(12,2) NOT NULL,
    CURRENT_UNIT_COST    NUMBER(12,2) NOT NULL,
    IS_ACTIVE            BOOLEAN NOT NULL,
    CONSTRAINT PK_PRODUCT_DIM PRIMARY KEY (PRODUCT_KEY)
);

CREATE OR REPLACE TABLE WAREHOUSE_DIM (
    WAREHOUSE_KEY        NUMBER(10,0) NOT NULL,
    WAREHOUSE_ID         VARCHAR(30) NOT NULL,
    WAREHOUSE_NAME       VARCHAR(150) NOT NULL,
    WAREHOUSE_TYPE       VARCHAR(30) NOT NULL,
    WAREHOUSE_REGION_KEY NUMBER(10,0) NOT NULL,
    CAPACITY_UNITS       NUMBER(12,0),
    IS_ACTIVE            BOOLEAN NOT NULL,
    CONSTRAINT PK_WAREHOUSE_DIM PRIMARY KEY (WAREHOUSE_KEY),
    CONSTRAINT FK_WAREHOUSE_REGION FOREIGN KEY (WAREHOUSE_REGION_KEY)
      REFERENCES REGION_DIM(REGION_KEY)
);

### Why region appears in customer and warehouse dimensions

Country and state describe both a customer's location and a warehouse's location. `REGION_DIM` gives those shared geographic values one consistent definition. `CUSTOMER_DIM.CUSTOMER_REGION_KEY` and `WAREHOUSE_DIM.WAREHOUSE_REGION_KEY` make it reusable.

For fast shipment-lane analysis, the fact also stores both resolved region keys. Analysts can join `REGION_DIM` twice with aliases such as `customer_region` and `warehouse_region`. These are role-playing uses of one conformed dimension.

## 4. Create the order-item fact table

In [ ]:
%%sql
CREATE OR REPLACE TABLE FACT_ORDER_ITEM (
    ORDER_ITEM_KEY       NUMBER(18,0) NOT NULL,

    -- Role-playing date keys
    ORDER_DATE_KEY       NUMBER(8,0) NOT NULL,
    SHIP_DATE_KEY        NUMBER(8,0) NOT NULL,

    -- Dimension keys
    CUSTOMER_KEY         NUMBER(10,0) NOT NULL,
    PRODUCT_KEY          NUMBER(10,0) NOT NULL,
    WAREHOUSE_KEY        NUMBER(10,0) NOT NULL,
    CUSTOMER_REGION_KEY  NUMBER(10,0) NOT NULL,
    WAREHOUSE_REGION_KEY NUMBER(10,0) NOT NULL,

    -- Degenerate dimensions
    ORDER_NUMBER         VARCHAR(30) NOT NULL,
    ORDER_LINE_NUMBER    NUMBER(6,0) NOT NULL,
    SALES_CHANNEL        VARCHAR(20) NOT NULL,
    PAYMENT_METHOD       VARCHAR(30) NOT NULL,
    ORDER_STATUS         VARCHAR(20) NOT NULL,
    PROMOTION_CODE       VARCHAR(30),

    -- Additive and derived measures at order-line grain
    QUANTITY             NUMBER(10,0) NOT NULL,
    UNIT_PRICE           NUMBER(12,2) NOT NULL,
    UNIT_COST            NUMBER(12,2) NOT NULL,
    GROSS_AMOUNT         NUMBER(14,2) NOT NULL,
    DISCOUNT_AMOUNT      NUMBER(14,2) NOT NULL,
    NET_AMOUNT           NUMBER(14,2) NOT NULL,
    TAX_AMOUNT           NUMBER(14,2) NOT NULL,
    SHIPPING_AMOUNT      NUMBER(14,2) NOT NULL,
    TOTAL_AMOUNT         NUMBER(14,2) NOT NULL,

    CONSTRAINT PK_FACT_ORDER_ITEM PRIMARY KEY (ORDER_ITEM_KEY),
    CONSTRAINT UQ_ORDER_LINE UNIQUE (ORDER_NUMBER, ORDER_LINE_NUMBER),
    CONSTRAINT FK_FACT_ORDER_DATE FOREIGN KEY (ORDER_DATE_KEY) REFERENCES DATE_DIM(DATE_KEY),
    CONSTRAINT FK_FACT_SHIP_DATE FOREIGN KEY (SHIP_DATE_KEY) REFERENCES DATE_DIM(DATE_KEY),
    CONSTRAINT FK_FACT_CUSTOMER FOREIGN KEY (CUSTOMER_KEY) REFERENCES CUSTOMER_DIM(CUSTOMER_KEY),
    CONSTRAINT FK_FACT_PRODUCT FOREIGN KEY (PRODUCT_KEY) REFERENCES PRODUCT_DIM(PRODUCT_KEY),
    CONSTRAINT FK_FACT_WAREHOUSE FOREIGN KEY (WAREHOUSE_KEY) REFERENCES WAREHOUSE_DIM(WAREHOUSE_KEY),
    CONSTRAINT FK_FACT_CUSTOMER_REGION FOREIGN KEY (CUSTOMER_REGION_KEY) REFERENCES REGION_DIM(REGION_KEY),
    CONSTRAINT FK_FACT_WAREHOUSE_REGION FOREIGN KEY (WAREHOUSE_REGION_KEY) REFERENCES REGION_DIM(REGION_KEY)
);

## 5. Populate the dimensions

The date dimension is generated for 2024–2026. A deterministic `ROW_NUMBER()` is used instead of relying on a gap-free sequence. The integer `YYYYMMDD` key is readable and stable; date arithmetic still uses `FULL_DATE`.

In [ ]:
%%sql
INSERT INTO DATE_DIM
SELECT
    TO_NUMBER(TO_CHAR(FULL_DATE, 'YYYYMMDD'))                    AS DATE_KEY,
    FULL_DATE,
    DAYOFWEEKISO(FULL_DATE)                                     AS DAY_OF_WEEK_NUMBER,
    DAYNAME(FULL_DATE)                                          AS DAY_NAME,
    DAY(FULL_DATE)                                              AS DAY_OF_MONTH,
    DAYOFYEAR(FULL_DATE)                                        AS DAY_OF_YEAR,
    WEEKISO(FULL_DATE)                                          AS WEEK_OF_YEAR,
    MONTH(FULL_DATE)                                            AS MONTH_NUMBER,
    MONTHNAME(FULL_DATE)                                        AS MONTH_NAME,
    QUARTER(FULL_DATE)                                          AS QUARTER_NUMBER,
    'Q' || QUARTER(FULL_DATE)                                   AS QUARTER_NAME,
    YEAR(FULL_DATE)                                             AS YEAR_NUMBER,
    IFF(DAYOFWEEKISO(FULL_DATE) IN (6, 7), TRUE, FALSE)          AS IS_WEEKEND
FROM (
    SELECT DATEADD(DAY, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1,
                   TO_DATE('2024-01-01')) AS FULL_DATE
    FROM TABLE(GENERATOR(ROWCOUNT => 1096))
)
WHERE FULL_DATE <= TO_DATE('2026-12-31');

INSERT INTO REGION_DIM
  (REGION_KEY, COUNTRY_CODE, COUNTRY_NAME, STATE_CODE, STATE_NAME, SALES_REGION)
VALUES
  (-1, 'NA', 'Unknown', 'NA', 'Unknown', 'Unknown'),
  (101, 'US', 'United States', 'CA', 'California', 'West'),
  (102, 'US', 'United States', 'NY', 'New York', 'Northeast'),
  (103, 'US', 'United States', 'TX', 'Texas', 'South'),
  (104, 'US', 'United States', 'IL', 'Illinois', 'Midwest'),
  (201, 'IN', 'India', 'KA', 'Karnataka', 'South India'),
  (202, 'IN', 'India', 'MH', 'Maharashtra', 'West India');

INSERT INTO CUSTOMER_DIM
  (CUSTOMER_KEY, CUSTOMER_ID, CUSTOMER_NAME, EMAIL, CUSTOMER_SEGMENT,
   CUSTOMER_REGION_KEY, EFFECTIVE_FROM, EFFECTIVE_TO, IS_CURRENT)
VALUES
  (-1, 'UNKNOWN', 'Unknown Customer', NULL, 'Unknown', -1, '1900-01-01', '9999-12-31', TRUE),
  (1001, 'C001', 'Ava Patel', 'ava@example.com', 'Consumer', 101, '2024-01-01', '9999-12-31', TRUE),
  (1002, 'C002', 'Noah Williams', 'noah@example.com', 'Corporate', 102, '2024-01-01', '9999-12-31', TRUE),
  (1003, 'C003', 'Mia Garcia', 'mia@example.com', 'Consumer', 103, '2024-01-01', '9999-12-31', TRUE),
  (1004, 'C004', 'Liam Shah', 'liam@example.com', 'Small Business', 201, '2024-01-01', '9999-12-31', TRUE),
  (1005, 'C005', 'Emma Johnson', 'emma@example.com', 'Corporate', 104, '2024-01-01', '9999-12-31', TRUE);

INSERT INTO PRODUCT_DIM
  (PRODUCT_KEY, PRODUCT_ID, SKU, PRODUCT_NAME, BRAND_NAME, CATEGORY_ID,
   CATEGORY_NAME, SUBCATEGORY_NAME, CURRENT_LIST_PRICE, CURRENT_UNIT_COST, IS_ACTIVE)
VALUES
  (-1, 'UNKNOWN', 'UNKNOWN', 'Unknown Product', 'Unknown', 'UNKNOWN', 'Unknown', 'Unknown', 0, 0, TRUE),
  (2001, 'P001', 'ELEC-LAP-01', 'Aurora 14 Laptop', 'Northstar', 'CAT-ELEC', 'Electronics', 'Computers', 1200, 820, TRUE),
  (2002, 'P002', 'ELEC-AUD-01', 'Wireless Headphones', 'SoundPeak', 'CAT-ELEC', 'Electronics', 'Audio', 180, 90, TRUE),
  (2003, 'P003', 'HOME-KIT-01', 'Smart Blender', 'HomeCraft', 'CAT-HOME', 'Home & Kitchen', 'Kitchen Appliances', 140, 72, TRUE),
  (2004, 'P004', 'SPORT-FIT-01', 'Fitness Tracker', 'PulseGo', 'CAT-SPORT', 'Sports & Fitness', 'Wearables', 110, 48, TRUE),
  (2005, 'P005', 'BOOK-DATA-01', 'Practical Data Warehousing', 'DataPress', 'CAT-BOOK', 'Books', 'Technology', 60, 22, TRUE),
  (2006, 'P006', 'HOME-OFF-01', 'Ergonomic Desk Chair', 'WorkWell', 'CAT-HOME', 'Home & Kitchen', 'Office Furniture', 320, 175, TRUE);

INSERT INTO WAREHOUSE_DIM
  (WAREHOUSE_KEY, WAREHOUSE_ID, WAREHOUSE_NAME, WAREHOUSE_TYPE,
   WAREHOUSE_REGION_KEY, CAPACITY_UNITS, IS_ACTIVE)
VALUES
  (-1, 'UNKNOWN', 'Unknown Warehouse', 'Unknown', -1, NULL, TRUE),
  (3001, 'W001', 'California Fulfilment Center', 'Fulfilment Center', 101, 500000, TRUE),
  (3002, 'W002', 'Texas Fulfilment Center', 'Fulfilment Center', 103, 350000, TRUE),
  (3003, 'W003', 'Karnataka Fulfilment Center', 'Fulfilment Center', 201, 400000, TRUE);

## 6. Insert order-item facts

These rows are already expressed at the declared analytical grain. In a production ELT process, source order and order-item records would be joined to current dimension rows to resolve surrogate keys before insertion.

The formulas used are:

```text
gross_amount = quantity × unit_price
net_amount   = gross_amount − discount_amount
total_amount = net_amount + tax_amount + shipping_amount
profit       = net_amount − (quantity × unit_cost)
```

In [ ]:
%%sql
INSERT INTO FACT_ORDER_ITEM
  (ORDER_ITEM_KEY, ORDER_DATE_KEY, SHIP_DATE_KEY, CUSTOMER_KEY, PRODUCT_KEY,
   WAREHOUSE_KEY, CUSTOMER_REGION_KEY, WAREHOUSE_REGION_KEY,
   ORDER_NUMBER, ORDER_LINE_NUMBER, SALES_CHANNEL, PAYMENT_METHOD,
   ORDER_STATUS, PROMOTION_CODE, QUANTITY, UNIT_PRICE, UNIT_COST,
   GROSS_AMOUNT, DISCOUNT_AMOUNT, NET_AMOUNT, TAX_AMOUNT,
   SHIPPING_AMOUNT, TOTAL_AMOUNT)
VALUES
  (1, 20250105, 20250106, 1001, 2001, 3001, 101, 101, 'ORD-1001', 1, 'Web', 'Credit Card', 'Shipped', 'NEWYEAR10', 1, 1200, 820, 1200, 120, 1080, 86.40, 0, 1166.40),
  (2, 20250105, 20250106, 1001, 2002, 3001, 101, 101, 'ORD-1001', 2, 'Web', 'Credit Card', 'Shipped', 'NEWYEAR10', 2, 180, 90, 360, 36, 324, 25.92, 0, 349.92),
  (3, 20250107, 20250108, 1002, 2003, 3002, 102, 103, 'ORD-1002', 1, 'Mobile', 'Digital Wallet', 'Delivered', NULL, 1, 140, 72, 140, 0, 140, 11.20, 12, 163.20),
  (4, 20250110, 20250111, 1003, 2004, 3002, 103, 103, 'ORD-1003', 1, 'Web', 'Debit Card', 'Delivered', 'FIT15', 2, 110, 48, 220, 33, 187, 14.96, 0, 201.96),
  (5, 20250115, 20250117, 1004, 2005, 3003, 201, 201, 'ORD-1004', 1, 'Marketplace', 'UPI', 'Delivered', NULL, 3, 60, 22, 180, 0, 180, 9, 5, 194),
  (6, 20250202, 20250203, 1005, 2006, 3001, 104, 101, 'ORD-1005', 1, 'Sales Rep', 'Invoice', 'Shipped', 'B2B20', 5, 320, 175, 1600, 320, 1280, 102.40, 80, 1462.40),
  (7, 20250202, 20250203, 1005, 2002, 3001, 104, 101, 'ORD-1005', 2, 'Sales Rep', 'Invoice', 'Shipped', 'B2B20', 5, 180, 90, 900, 180, 720, 57.60, 0, 777.60),
  (8, 20250208, 20250209, 1002, 2001, 3001, 102, 101, 'ORD-1006', 1, 'Mobile', 'Credit Card', 'Delivered', NULL, 1, 1150, 820, 1150, 0, 1150, 92, 0, 1242),
  (9, 20250214, 20250215, 1001, 2004, 3002, 101, 103, 'ORD-1007', 1, 'Web', 'Digital Wallet', 'Delivered', 'LOVE10', 1, 110, 48, 110, 11, 99, 7.92, 8, 114.92),
  (10, 20250301, 20250303, 1003, 2003, 3002, 103, 103, 'ORD-1008', 1, 'Marketplace', 'Credit Card', 'Shipped', NULL, 2, 135, 72, 270, 0, 270, 21.60, 10, 301.60),
  (11, 20250301, 20250303, 1003, 2005, 3002, 103, 103, 'ORD-1008', 2, 'Marketplace', 'Credit Card', 'Shipped', NULL, 1, 60, 22, 60, 0, 60, 4.80, 0, 64.80),
  (12, 20250312, 20250313, 1004, 2002, 3003, 201, 201, 'ORD-1009', 1, 'Mobile', 'UPI', 'Delivered', 'SPRING5', 2, 175, 90, 350, 17.50, 332.50, 16.63, 0, 349.13);

## 7. Inspect the star through a reusable semantic view

The view hides repeated joins and gives BI tools friendly column names. Joining the date and region dimensions twice illustrates role-playing dimensions.

In [ ]:
%%sql
CREATE OR REPLACE VIEW VW_ORDER_ITEM_ANALYTICS AS
SELECT
    f.ORDER_ITEM_KEY,
    f.ORDER_NUMBER,
    f.ORDER_LINE_NUMBER,
    od.FULL_DATE                                        AS ORDER_DATE,
    sd.FULL_DATE                                        AS SHIP_DATE,
    DATEDIFF(DAY, od.FULL_DATE, sd.FULL_DATE)            AS DAYS_TO_SHIP,
    c.CUSTOMER_ID,
    c.CUSTOMER_NAME,
    c.CUSTOMER_SEGMENT,
    cr.COUNTRY_NAME                                     AS CUSTOMER_COUNTRY,
    cr.STATE_NAME                                       AS CUSTOMER_STATE,
    cr.SALES_REGION                                     AS CUSTOMER_SALES_REGION,
    p.SKU,
    p.PRODUCT_NAME,
    p.BRAND_NAME,
    p.CATEGORY_NAME,
    p.SUBCATEGORY_NAME,
    w.WAREHOUSE_ID,
    w.WAREHOUSE_NAME,
    wr.COUNTRY_NAME                                     AS WAREHOUSE_COUNTRY,
    wr.STATE_NAME                                       AS WAREHOUSE_STATE,
    f.SALES_CHANNEL,
    f.PAYMENT_METHOD,
    f.ORDER_STATUS,
    f.PROMOTION_CODE,
    f.QUANTITY,
    f.UNIT_PRICE,
    f.GROSS_AMOUNT,
    f.DISCOUNT_AMOUNT,
    f.NET_AMOUNT,
    f.TAX_AMOUNT,
    f.SHIPPING_AMOUNT,
    f.TOTAL_AMOUNT,
    f.NET_AMOUNT - (f.QUANTITY * f.UNIT_COST)            AS GROSS_PROFIT
FROM FACT_ORDER_ITEM f
JOIN DATE_DIM od      ON f.ORDER_DATE_KEY = od.DATE_KEY
JOIN DATE_DIM sd      ON f.SHIP_DATE_KEY = sd.DATE_KEY
JOIN CUSTOMER_DIM c   ON f.CUSTOMER_KEY = c.CUSTOMER_KEY
JOIN PRODUCT_DIM p    ON f.PRODUCT_KEY = p.PRODUCT_KEY
JOIN WAREHOUSE_DIM w  ON f.WAREHOUSE_KEY = w.WAREHOUSE_KEY
JOIN REGION_DIM cr    ON f.CUSTOMER_REGION_KEY = cr.REGION_KEY
JOIN REGION_DIM wr    ON f.WAREHOUSE_REGION_KEY = wr.REGION_KEY;

SELECT *
FROM VW_ORDER_ITEM_ANALYTICS
ORDER BY ORDER_NUMBER, ORDER_LINE_NUMBER;

## 8. Analytical query examples

Because all dimensions join directly to the fact, queries remain predictable: start with the fact, join dimensions needed for labels, aggregate measures, and group by dimension attributes.

### Monthly revenue and profit by category

`NET_AMOUNT` is used as merchandise revenue. `TOTAL_AMOUNT` also includes tax and shipping collected from customers.

In [ ]:
%%sql
SELECT
    d.YEAR_NUMBER,
    d.MONTH_NUMBER,
    d.MONTH_NAME,
    p.CATEGORY_NAME,
    SUM(f.QUANTITY)                                      AS UNITS_SOLD,
    ROUND(SUM(f.GROSS_AMOUNT), 2)                        AS GROSS_SALES,
    ROUND(SUM(f.DISCOUNT_AMOUNT), 2)                     AS DISCOUNTS,
    ROUND(SUM(f.NET_AMOUNT), 2)                          AS NET_REVENUE,
    ROUND(SUM(f.NET_AMOUNT - f.QUANTITY * f.UNIT_COST), 2) AS GROSS_PROFIT
FROM FACT_ORDER_ITEM f
JOIN DATE_DIM d    ON f.ORDER_DATE_KEY = d.DATE_KEY
JOIN PRODUCT_DIM p ON f.PRODUCT_KEY = p.PRODUCT_KEY
GROUP BY d.YEAR_NUMBER, d.MONTH_NUMBER, d.MONTH_NAME, p.CATEGORY_NAME
ORDER BY d.YEAR_NUMBER, d.MONTH_NUMBER, NET_REVENUE DESC;

### Customer region to warehouse region shipment lanes

In [ ]:
%%sql
SELECT
    cr.COUNTRY_NAME || ' / ' || cr.STATE_NAME AS CUSTOMER_LOCATION,
    wr.COUNTRY_NAME || ' / ' || wr.STATE_NAME AS WAREHOUSE_LOCATION,
    COUNT(DISTINCT f.ORDER_NUMBER)             AS ORDERS,
    SUM(f.QUANTITY)                            AS UNITS,
    ROUND(SUM(f.NET_AMOUNT), 2)                AS NET_REVENUE
FROM FACT_ORDER_ITEM f
JOIN REGION_DIM cr ON f.CUSTOMER_REGION_KEY = cr.REGION_KEY
JOIN REGION_DIM wr ON f.WAREHOUSE_REGION_KEY = wr.REGION_KEY
GROUP BY CUSTOMER_LOCATION, WAREHOUSE_LOCATION
ORDER BY NET_REVENUE DESC;

### Customer performance and average order value

In [ ]:
%%sql
SELECT
    c.CUSTOMER_SEGMENT,
    COUNT(DISTINCT f.CUSTOMER_KEY)                      AS ACTIVE_CUSTOMERS,
    COUNT(DISTINCT f.ORDER_NUMBER)                      AS ORDER_COUNT,
    SUM(f.QUANTITY)                                     AS UNITS,
    ROUND(SUM(f.NET_AMOUNT), 2)                         AS NET_REVENUE,
    ROUND(SUM(f.NET_AMOUNT) / NULLIF(COUNT(DISTINCT f.ORDER_NUMBER), 0), 2)
                                                         AS AVG_ORDER_VALUE
FROM FACT_ORDER_ITEM f
JOIN CUSTOMER_DIM c ON f.CUSTOMER_KEY = c.CUSTOMER_KEY
GROUP BY c.CUSTOMER_SEGMENT
ORDER BY NET_REVENUE DESC;

### Product ranking within category

In [ ]:
%%sql
WITH PRODUCT_SALES AS (
    SELECT
        p.CATEGORY_NAME,
        p.PRODUCT_NAME,
        SUM(f.QUANTITY) AS UNITS_SOLD,
        SUM(f.NET_AMOUNT) AS NET_REVENUE
    FROM FACT_ORDER_ITEM f
    JOIN PRODUCT_DIM p ON f.PRODUCT_KEY = p.PRODUCT_KEY
    GROUP BY p.CATEGORY_NAME, p.PRODUCT_NAME
)
SELECT
    CATEGORY_NAME,
    PRODUCT_NAME,
    UNITS_SOLD,
    ROUND(NET_REVENUE, 2) AS NET_REVENUE,
    DENSE_RANK() OVER (
        PARTITION BY CATEGORY_NAME ORDER BY NET_REVENUE DESC
    ) AS CATEGORY_REVENUE_RANK
FROM PRODUCT_SALES
ORDER BY CATEGORY_NAME, CATEGORY_REVENUE_RANK;

### Promotion effectiveness

In [ ]:
%%sql
SELECT
    COALESCE(PROMOTION_CODE, 'NO PROMOTION') AS PROMOTION,
    COUNT(DISTINCT ORDER_NUMBER)             AS ORDERS,
    SUM(QUANTITY)                            AS UNITS,
    ROUND(SUM(GROSS_AMOUNT), 2)              AS GROSS_SALES,
    ROUND(SUM(DISCOUNT_AMOUNT), 2)           AS DISCOUNT_GIVEN,
    ROUND(SUM(NET_AMOUNT), 2)                AS NET_REVENUE,
    ROUND(100 * SUM(DISCOUNT_AMOUNT) / NULLIF(SUM(GROSS_AMOUNT), 0), 2)
                                               AS DISCOUNT_RATE_PCT
FROM FACT_ORDER_ITEM
GROUP BY PROMOTION
ORDER BY NET_REVENUE DESC;

## 9. Data-quality checks

On Snowflake standard tables, declared primary, unique, and foreign keys are descriptive metadata rather than fully enforced relational constraints. Reliable warehouse loads therefore test these assumptions explicitly.

Every query below should return zero rows, except the reconciliation query, which should return a single `PASS` row.

In [ ]:
%%sql
-- Duplicate business grain: must return zero rows.
SELECT ORDER_NUMBER, ORDER_LINE_NUMBER, COUNT(*) AS ROW_COUNT
FROM FACT_ORDER_ITEM
GROUP BY ORDER_NUMBER, ORDER_LINE_NUMBER
HAVING COUNT(*) > 1;

In [ ]:
%%sql
-- Orphaned dimension keys: must return zero rows.
SELECT f.ORDER_ITEM_KEY,
       IFF(od.DATE_KEY IS NULL, 'MISSING ORDER DATE; ', '') ||
       IFF(sd.DATE_KEY IS NULL, 'MISSING SHIP DATE; ', '') ||
       IFF(c.CUSTOMER_KEY IS NULL, 'MISSING CUSTOMER; ', '') ||
       IFF(p.PRODUCT_KEY IS NULL, 'MISSING PRODUCT; ', '') ||
       IFF(w.WAREHOUSE_KEY IS NULL, 'MISSING WAREHOUSE; ', '') ||
       IFF(cr.REGION_KEY IS NULL, 'MISSING CUSTOMER REGION; ', '') ||
       IFF(wr.REGION_KEY IS NULL, 'MISSING WAREHOUSE REGION; ', '') AS ISSUE
FROM FACT_ORDER_ITEM f
LEFT JOIN DATE_DIM od ON f.ORDER_DATE_KEY = od.DATE_KEY
LEFT JOIN DATE_DIM sd ON f.SHIP_DATE_KEY = sd.DATE_KEY
LEFT JOIN CUSTOMER_DIM c ON f.CUSTOMER_KEY = c.CUSTOMER_KEY
LEFT JOIN PRODUCT_DIM p ON f.PRODUCT_KEY = p.PRODUCT_KEY
LEFT JOIN WAREHOUSE_DIM w ON f.WAREHOUSE_KEY = w.WAREHOUSE_KEY
LEFT JOIN REGION_DIM cr ON f.CUSTOMER_REGION_KEY = cr.REGION_KEY
LEFT JOIN REGION_DIM wr ON f.WAREHOUSE_REGION_KEY = wr.REGION_KEY
WHERE od.DATE_KEY IS NULL OR sd.DATE_KEY IS NULL OR c.CUSTOMER_KEY IS NULL
   OR p.PRODUCT_KEY IS NULL OR w.WAREHOUSE_KEY IS NULL
   OR cr.REGION_KEY IS NULL OR wr.REGION_KEY IS NULL;

In [ ]:
%%sql
-- Measure reconciliation: expected result is PASS.
SELECT
    IFF(
        COUNT_IF(GROSS_AMOUNT <> QUANTITY * UNIT_PRICE) = 0
        AND COUNT_IF(NET_AMOUNT <> GROSS_AMOUNT - DISCOUNT_AMOUNT) = 0
        AND COUNT_IF(TOTAL_AMOUNT <> NET_AMOUNT + TAX_AMOUNT + SHIPPING_AMOUNT) = 0,
        'PASS', 'FAIL'
    ) AS MEASURE_RECONCILIATION,
    COUNT(*) AS FACT_ROWS
FROM FACT_ORDER_ITEM;

## 10. Conformed and role-playing dimensions

A reusable dimension shared across business processes, fact tables, or data marts is formally called a **conformed dimension**. It supplies the same keys, attribute definitions, and business meaning everywhere, allowing results from different processes to be compared consistently. A single `DIM_DATE`, `DIM_CUSTOMER`, or `DIM_PRODUCT` can therefore support facts such as sales, inventory, returns, and shipments.

A **role-playing dimension** is one physical dimension reused more than once in a fact table, with each foreign key giving it a different business meaning. The table is not copied. Queries join it multiple times using descriptive aliases, and a semantic layer may expose role-specific logical views.

### How these concepts appear in this model

| Physical conformed dimension | Foreign-key role in `FACT_ORDER_ITEM` | Query alias / logical name | Meaning |
|---|---|---|---|
| `DATE_DIM` | `ORDER_DATE_KEY` | `order_date` | Date on which the customer placed the order |
| `DATE_DIM` | `SHIP_DATE_KEY` | `ship_date` | Date on which the warehouse shipped the line |
| `REGION_DIM` | `CUSTOMER_REGION_KEY` | `customer_region` | Customer's country, state, and sales region |
| `REGION_DIM` | `WAREHOUSE_REGION_KEY` | `warehouse_region` | Fulfilling warehouse's country, state, and sales region |

`DATE_DIM` and `REGION_DIM` are conformed base tables because their definitions are reusable. Within `FACT_ORDER_ITEM`, each also plays multiple roles. Conformed describes consistency and reuse across the warehouse; role-playing describes the context assigned to repeated references in a particular model. A dimension can be both.

### Naming conventions

Physical tables normally use an entity name such as `DIM_DATE` or this notebook's `DATE_DIM`. Role names should communicate business context, such as `ORDER_DATE`, `SHIP_DATE`, `CUSTOMER_REGION`, and `WAREHOUSE_REGION`. Clear names prevent an analyst from joining a valid dimension through the wrong foreign key.

### Physical table, aliases, and optional views

SQL aliases are usually sufficient inside a query. The following optional views expose business-friendly role names without duplicating any data. They are useful when a BI or semantic tool expects a separate logical object per role.

In [ ]:
%%sql
CREATE OR REPLACE VIEW DIM_ORDER_DATE AS
SELECT * FROM DATE_DIM;

CREATE OR REPLACE VIEW DIM_SHIP_DATE AS
SELECT * FROM DATE_DIM;

CREATE OR REPLACE VIEW DIM_CUSTOMER_REGION AS
SELECT * FROM REGION_DIM;

CREATE OR REPLACE VIEW DIM_WAREHOUSE_REGION AS
SELECT * FROM REGION_DIM;

The views above contain no copied rows. They are logical names over the same physical conformed dimensions. For direct SQL, aliases remain concise:

```sql
SELECT od.FULL_DATE AS ORDER_DATE, sd.FULL_DATE AS SHIP_DATE
FROM FACT_ORDER_ITEM f
JOIN DATE_DIM od ON f.ORDER_DATE_KEY = od.DATE_KEY
JOIN DATE_DIM sd ON f.SHIP_DATE_KEY = sd.DATE_KEY;
```

### Further reading

- [Microsoft Fabric: Dimension tables in a warehouse](https://learn.microsoft.com/en-us/fabric/data-warehouse/dimensional-modeling-dimension-tables)
- [ScienceDirect topic: Conformed dimension](https://www.sciencedirect.com/topics/computer-science/conformed-dimension)
- [Wikipedia: Dimension — role-playing dimension](https://en.wikipedia.org/wiki/Dimension_%28data_warehouse%29)

## 11. How this model evolves in production

### Slowly changing dimensions

`CUSTOMER_DIM` includes `EFFECTIVE_FROM`, `EFFECTIVE_TO`, and `IS_CURRENT`, so it can become a Type 2 slowly changing dimension. When a customer changes segment or region, expire the current row and insert a new surrogate-keyed row. Existing facts retain the old customer key and therefore the historically correct attributes.

Product attributes can use the same approach when historical category or brand reporting is required. If only the latest description matters, update the dimension row in place (Type 1).

### Late-arriving data

Load a fact with key `-1` when its dimension record is unavailable. After the dimension arrives, resolve and update that fact key through a controlled correction process. Never discard the business event merely because descriptive data is late.

### Performance

Snowflake generally performs well without user-managed indexes. Start with clear grain, correct joins, selective queries, and an appropriately sized warehouse. For a very large fact table, consider clustering only after query-profile evidence shows repeated pruning problems—commonly on order date or another dominant filter.

### Semantic correctness

- Sum line measures such as quantity and net revenue.
- Use `COUNT(DISTINCT ORDER_NUMBER)` for orders because one order spans multiple rows.
- Do not sum `UNIT_PRICE`; aggregate the extended amounts.
- Use order date for demand analysis and ship date for fulfilment analysis.
- Join the same conformed dimension with clear aliases for each role.

This model is a complete analytical star: a central order-item fact, denormalized business dimensions, conformed date and region dimensions, sample records, reusable view, analysis examples, and warehouse-specific quality checks.